# Benchmark — harness reprodutível

Executa um **experimento controlado** do caminho arquitetural central (streaming Delta
+ `foreachBatch` MERGE) para um dado volume e configuração de cluster, mede a duração
e o throughput reais e **registra o resultado** em uma tabela.

Rode uma vez por cenário (variando `volume` e o tamanho do cluster no bundle) e
compare as linhas acumuladas em `__benchmark_results`. Os números vêm da execução
real — não há valores pré-preenchidos.

## Parâmetros

In [ ]:
import sys, time
sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("scenario", "A")
dbutils.widgets.text("volume", "1000000")
dbutils.widgets.text("num_workers", "1")
dbutils.widgets.text("node_type", "Standard_D4ds_v6")
dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("schema", "s_dm_callcenter")

SCENARIO    = dbutils.widgets.get("scenario")
VOLUME      = int(dbutils.widgets.get("volume"))
NUM_WORKERS = int(dbutils.widgets.get("num_workers"))
NODE_TYPE   = dbutils.widgets.get("node_type")
CATALOG     = dbutils.widgets.get("catalog")
SCHEMA      = dbutils.widgets.get("schema")

BRONZE_FQN  = f"{CATALOG}.{SCHEMA}.__bench_bronze"
SILVER_FQN  = f"{CATALOG}.{SCHEMA}.__bench_silver"
RESULTS_FQN = f"{CATALOG}.{SCHEMA}.__benchmark_results"
CHECKPOINT  = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/benchmark/{SCENARIO}"
print(f"cenário={SCENARIO} volume={VOLUME:,} workers={NUM_WORKERS} node={NODE_TYPE}")

## Carga sintética (Bronze estruturada, append-only)

In [ ]:
from pyspark.sql import functions as F

# Bronze sintética JÁ ESTRUTURADA, no mesmo formato que a Bronze de produção entrega
# à Silver (campos parseados). O SilverStream consome sem refazer parse.
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_FQN}")
dbutils.fs.rm(CHECKPOINT, recurse=True)

gen = (
    spark.range(VOLUME)
    .select(
        F.col("id").cast("string").alias("id_chamada"),
        (F.col("id") % 5000).cast("string").alias("id_cliente"),
        F.current_timestamp().alias("data_hora_inicio"),
    )
)
gen.write.format("delta").mode("overwrite").saveAsTable(BRONZE_FQN)
print(f"[OK] Bronze sintética (estruturada): {VOLUME:,} eventos")

## Execução medida (stream Delta → foreachBatch MERGE)

In [ ]:
from transforms import SilverStream, rename_columns


def transform(df):
    # A Bronze já entrega estruturado; a Silver só projeta/renomeia.
    return rename_columns(df, {"id_chamada": "ID_CHAM", "id_cliente": "ID_CLIE", "data_hora_inicio": "DH_INIC"})


spark.sql(f"DROP TABLE IF EXISTS {SILVER_FQN}")

start = time.time()
SilverStream(spark).run(
    source_table_fqn=BRONZE_FQN,
    target_table_fqn=SILVER_FQN,
    transform=transform,
    keys=["ID_CHAM"],
    checkpoint_location=CHECKPOINT,
)
duration_s = round(time.time() - start, 2)

output_rows = spark.table(SILVER_FQN).count()
throughput = round(output_rows / duration_s, 1) if duration_s > 0 else None
print(f"duração={duration_s}s  linhas={output_rows:,}  throughput={throughput} rows/s")

## Registro do resultado

In [ ]:
cols = ["scenario", "volume", "num_workers", "node_type",
        "duration_s", "throughput_rows_s", "input_rows", "output_rows"]
row = [(SCENARIO, VOLUME, NUM_WORKERS, NODE_TYPE, duration_s, throughput, VOLUME, output_rows)]

df = spark.createDataFrame(row, cols).withColumn("run_at", F.current_timestamp())
df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(RESULTS_FQN)

print("Resultados acumulados:")
display(spark.table(RESULTS_FQN).orderBy("scenario", "volume", "num_workers"))